# Machine Health — features físicas y validación por cambio de régimen

Versión 3. Reemplaza a `eda_y_benchmark.ipynb`, cuyo mejor envío dio **43,42**
en el servidor contra un CV de ~48. Este notebook existe para explicar esa
brecha y cerrarla.

**Los tres cambios respecto de la versión anterior**, en orden de importancia:

1. **Una segunda validación, por cambio de régimen.** `GroupKFold` por máquina
   mide *"máquina nueva, mismo régimen"*, pero el test es *"máquina nueva,
   régimen nuevo"*. Es la causa de la brecha, y ahora se mide.
2. **Features físicas adimensionales.** Ratios normalizados por velocidad de
   giro, desbalances entre fases, diferencias térmicas contra ambiente y leyes
   de afinidad de bombas. Es la palanca real.
3. **Promedio por sesión.** Cada sesión tiene 7 ventanas y la etiqueta es
   constante en las 250. El tamaño efectivo de la muestra es **250 sesiones,
   no 1750 ventanas**.

Alcance: sólo datos tabulares. **El procesamiento de NPZ queda fuera** de esta
etapa; el objetivo es agotar el techo tabular primero.

---

### Resumen de lo medido

| Features | GroupKFold | Cambio de régimen |
|---|---|---|
| crudas | 44,01 | 37,56 |
| crudas + z-score ← *versión anterior* | 46,65 | 40,53 |
| físicas | 49,37 | 40,67 |
| **físicas + z-score** | **51,76** | **42,57** |

| Ensamble (sobre físicas + z) | GroupKFold | + sesión | Régimen | + sesión |
|---|---|---|---|---|
| lgb | 51,76 | 51,70 | 42,57 | 43,63 |
| lgb + rf + lr ← *recomendación anterior* | 53,77 | **54,44** | 44,25 | 45,43 |
| **lgb + et + rf** | 52,18 | 52,72 | **47,44** | **47,43** |

La configuración que gana bajo `GroupKFold` **no** es la que gana bajo cambio de
régimen, y la que ganaba antes (`lgb+rf+lr`) es de las peores en el esquema que
reproduce la condición real del test. Se elige `lgb+et+rf`.

## 0. Entorno y reproducibilidad

La consigna pide adjuntar el código que replica la solución enviada (art. 5).
Eso obliga a un detalle que es fácil pasar por alto: **`ExtraTrees` y
`RandomForest` con `n_jobs=-1` no son deterministas aunque se fije
`random_state`**. La celda de verificación lo comprueba.

In [ ]:
# Colab: descomentar.
# !pip -q install lightgbm pyarrow

import os, sys, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
N_JOBS = 1          # NO cambiar a -1: rompe la reproducibilidad del submit

DATA = Path("datos_sinraw")        # train_sup.parquet, test.parquet
KIT  = Path("participant_kit")     # scoring.py de la cátedra
sys.path.insert(0, str(KIT)); sys.path.insert(0, ".")

import scoring
from scoring import compute_score, validate_prediction_package
from scoring import FAULT_IDS, LABEL_COLUMNS, FAMILIES

train = pd.read_parquet(DATA / "train_sup.parquet")
test  = pd.read_parquet(DATA / "test.parquet")
print("train", train.shape, "| test", test.shape)

In [ ]:
# Verificación del determinismo. Si esto falla, el submit no se puede replicar.
from sklearn.ensemble import ExtraTreesClassifier

_lab = train[LABEL_COLUMNS].to_numpy()
estado = np.where(_lab.sum(axis=1) == 0, 0, _lab.argmax(axis=1) + 1)
ESTADOS = ["sano"] + FAULT_IDS

_X = train.select_dtypes(include=[np.number]).drop(
    columns=[c for c in train.columns if c.startswith(("label_", "severity_"))],
    errors="ignore").fillna(-999)

def _hash(n_jobs, n_est=600):
    m = ExtraTreesClassifier(n_estimators=n_est, n_jobs=n_jobs,
                             random_state=RANDOM_STATE).fit(_X, estado)
    return hashlib.md5(m.predict_proba(_X).tobytes()).hexdigest()[:12]

a1, a2 = _hash(1), _hash(1)
b1, b2 = _hash(-1), _hash(-1)
print("n_jobs=1  ->", a1, a2, "IGUALES" if a1 == a2 else "DISTINTOS  <-- problema")
print("n_jobs=-1 ->", b1, b2, "iguales (esta corrida)" if b1 == b2 else "DISTINTOS  <-- no reproducible")
assert a1 == a2, "ni con n_jobs=1 es determinista: revisar el entorno"
print("""
Con n_jobs=-1 el resultado depende de cómo se repartan los árboles entre hilos,
así que puede coincidir por casualidad en una corrida y no en la siguiente. Con
n_jobs=1 no depende de nada: es la única forma de garantizar el art. 5.""")

## 1. EDA dirigido

Tres preguntas, no un EDA genérico: ¿está limpio?, ¿cuál es la estructura real
del target?, y ¿cuál es el tamaño efectivo de la muestra?

In [ ]:
print("nulos          :", int(train.isna().sum().sum()))
print("duplicados     :", int(train.duplicated().sum()))
print("constantes     :", [c for c in train.columns if train[c].nunique(dropna=False) <= 1])
print("centinelas -999:", int((train.select_dtypes(include=[np.number]) == -999).sum().sum()))
print("\nNo hay nada que curar. El trabajo está en las features, no en la limpieza.")

### Estructura del target y tamaño efectivo de la muestra

Este es el hallazgo que más cambia el diseño de la validación.

In [ ]:
n_fallas = _lab.sum(axis=1)
print("fallas por ventana:", pd.Series(n_fallas).value_counts().sort_index().to_dict())
assert train.is_combo.sum() == 0
print("-> 14 estados mutuamente excluyentes (sano + F01..F13), no 13 binarios\n")

vent = train.groupby("session_id").size()
etiq = train.assign(e=estado).groupby("session_id").e.nunique()
print("ventanas por sesión:", vent.value_counts().to_dict())
print("sesiones con etiqueta constante: %d de %d" % ((etiq == 1).sum(), len(etiq)))
print("\nTAMAÑO EFECTIVO = %d sesiones, no %d ventanas."
      % (train.session_id.nunique(), len(train)))
print("El riesgo de sobreajuste es 7 veces mayor de lo que sugiere el n nominal.")

In [ ]:
prev = (pd.Series(estado).value_counts().sort_index()
        .rename(index=lambda k: ESTADOS[k]).to_frame("n"))
prev["prevalencia"] = prev.n / len(train)
prev["familia"] = ["sano"] + [FAMILIES[f] for f in FAULT_IDS]
display(prev.T)

COL = {"sano": "#2E6B45", "mechanical": "#0E4F5C", "structural": "#6B7A83",
       "electrical": "#9A5B00", "hydraulic": "#3D7B8F"}
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(prev.index, prev.prevalencia, color=[COL[f] for f in prev.familia])
ax.set_ylabel("prevalencia"); ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Prevalencia por estado")
plt.tight_layout(); plt.show()

### Qué elige realmente el baseline de la cátedra

El `baseline.ipynb` del kit dice que "la acción óptima para probabilidades bajas
es MONITOR". Con sus propias prevalencias eso es falso, y conviene saberlo
porque fija la vara real que hay que superar.

In [ ]:
p_prev = {f: float(train["label_" + f].mean()) for f in FAULT_IDS}
S = sum(p_prev.values())
S_mec = sum(v for k, v in p_prev.items() if FAMILIES[k] == "mechanical")

print("S = %.3f   S_mecánica = %.3f" % (S, S_mec))
print("MONITOR       = %.2f" % (S * 10))
print("INSPECT mec.  = %.2f" % (4 + 8 * (S - S_mec)))
print("STOP          = 12.00")
print("choose_action ->", scoring.choose_action(p_prev))
print("""
El baseline NO es pasivo: inspecciona mecánica en las 1848 ventanas. Para
superarlo hay que acertar la familia más del 46 % de las veces.""")

## 2. Los dos esquemas de validación

Acá está la explicación de la brecha CV≈48 vs servidor 43,42.

`StratifiedGroupKFold` por `machine_id` garantiza que ninguna máquina esté a los
dos lados de la partición. Eso evita la fuga, pero mide **"máquina nueva, mismo
régimen"**: los folds de validación tienen máquinas lentas y rápidas mezcladas,
igual que los de ajuste.

El test es otra cosa. Sus 22 máquinas son todas distintas a las 26 del train
**y** operan en otro punto de trabajo.

In [ ]:
comp = pd.DataFrame({
    "train": train[["rpm_mean", "flow_mean", "Tamb", "delta_p_mean"]].mean(),
    "test":  test[["rpm_mean", "flow_mean", "Tamb", "delta_p_mean"]].mean()})
comp["cociente"] = comp.test / comp.train
display(comp.round(2))

rpm_maq = train.groupby("machine_id").rpm_mean.mean().sort_values()
LENTAS, RAPIDAS = set(rpm_maq.index[:13]), set(rpm_maq.index[13:])
i_lentas  = np.where(train.machine_id.isin(LENTAS))[0]
i_rapidas = np.where(train.machine_id.isin(RAPIDAS))[0]

print("13 máquinas lentas : rpm medio = %.0f" % train.rpm_mean.iloc[i_lentas].mean())
print("13 máquinas rápidas: rpm medio = %.0f" % train.rpm_mean.iloc[i_rapidas].mean())
print("test               : rpm medio = %.0f  <- se parece a las rápidas" % test.rpm_mean.mean())

`delta_p_mean` es el caso extremo: **5 veces más grande en el test**. Cualquier
feature que use la presión diferencial en valor absoluto está condenada.

De ahí el segundo esquema: **entrenar con las 13 máquinas lentas y validar con
las 13 rápidas**. Reproduce la relación real entre train y test. Es un solo
corte, así que es ruidoso, pero es el único estimador honesto que tenemos.

Cómo leer los dos números de acá en adelante:

- `GroupKFold` es el **optimista**: promedia 3 semillas, es estable, y sobrestima.
- Cambio de régimen es el **conservador**: un solo corte, ruidoso, y subestima.
- El servidor cayó **entre los dos** en el único caso que pudimos anclar:
  crudas+z dio 46,65 / 40,53 y el servidor 43,42.

## 3. Feature engineering con dominio físico

La regla que ordena todo este bloque: **adimensional o relativo**. Una feature
que depende del punto de trabajo no transfiere a un régimen nuevo; una que es un
cociente entre magnitudes que escalan igual, sí.

- **Vibración.** La energía vibratoria escala con el cuadrado de la velocidad,
  de ahí `rms/(rpm/1000)²`. Sin eso el modelo confunde *máquina rápida* con
  *máquina en falla*, que es exactamente el shift de la sección 2. La fracción
  de energía en `1x` marca **desbalance (F01)**; `2x/1x`, **desalineación
  (F02)**, porque la desalineación excita el segundo armónico. `log1p(kurtosis)`
  y `crest` capturan los impactos de **rodamientos (F03–F05)**. La asimetría
  entre apoyos marca **pérdida de rigidez (F07)**.
- **Eléctricas.** Desbalance de tensión y corriente como `(max−min)/media` para
  **F08**; `I_min/I_media` para **pérdida de fase (F09)**; corriente relativa a
  la velocidad para **barras rotóricas (F10)**.
- **Térmicas.** Siempre como diferencia contra `Tamb`, nunca en valor absoluto,
  porque el ambiente del test es 3 °C más cálido. Sobretemperatura de devanado
  normalizada por potencia → **cortocircuito entre espiras (F11)**; diferencia
  entre rodamientos → **lubricación deficiente (F06)**.
- **Hidráulicas.** Leyes de afinidad de bombas: altura adimensional
  `Δp/(rpm/1000)²` y coeficiente de caudal `flow/rpm`. El rendimiento aparente
  `flow·Δp/(V·I)` marca **obstrucción o fuga (F13)**; la presión de entrada
  normalizada, **cavitación (F12)**.

In [ ]:
EXCL = set(["window_id", "machine_id", "session_id", "is_normal", "is_combo"]
           + LABEL_COLUMNS + [f"severity_{f}" for f in FAULT_IDS])
BASE = [c for c in train.columns if c not in EXCL]

def fisicas(df):
    """Features adimensionales o relativas: la condición para transferir régimen."""
    d = pd.DataFrame(index=df.index)
    r  = df.rpm_mean / 1000.0
    r2 = (r ** 2).replace(0, np.nan)

    for ch in ["acc_radial_a", "acc_radial_b", "acc_axial"]:
        rms = df[f"{ch}__rms"]
        d[f"{ch}__rms_n"]    = rms / r2                                        # energía por régimen
        d[f"{ch}__1x_frac"]  = df[f"{ch}__1x"] / (df[f"{ch}__1x"] + df[f"{ch}__2x"] + 1e-9)
        d[f"{ch}__2x_1x"]    = df[f"{ch}__2x"] / (df[f"{ch}__1x"] + 1e-9)      # F02
        d[f"{ch}__1x_rms"]   = df[f"{ch}__1x"] / (rms + 1e-9)                  # F01
        d[f"{ch}__kurt_l"]   = np.log1p(df[f"{ch}__kurtosis"].clip(lower=0))   # F03-F05
        d[f"{ch}__crest_n"]  = df[f"{ch}__crest"]
        d[f"{ch}__kurt_rms"] = df[f"{ch}__kurtosis"] / (rms + 1e-9)
    d["acc_ab_ratio"] = df.acc_radial_a__rms / (df.acc_radial_b__rms + 1e-9)   # F07
    d["acc_ab_dif_n"] = (df.acc_radial_a__rms - df.acc_radial_b__rms) / r2
    d["acc_ax_rad"]   = df.acc_axial__rms / (df.acc_radial_a__rms + df.acc_radial_b__rms + 1e-9)
    d["acc_tot_n"]    = (df.acc_radial_a__rms + df.acc_radial_b__rms + df.acc_axial__rms) / r2

    V = df[["voltage_a__rms", "voltage_b__rms", "voltage_c__rms"]]
    I = df[["current_a__rms", "current_b__rms", "current_c__rms"]]
    d["V_unbal"]   = (V.max(1) - V.min(1)) / (V.mean(1) + 1e-9)                # F08
    d["I_unbal"]   = (I.max(1) - I.min(1)) / (I.mean(1) + 1e-9)
    d["I_min_rel"] = I.min(1) / (I.mean(1) + 1e-9)                             # F09
    d["I_max_rel"] = I.max(1) / (I.mean(1) + 1e-9)
    d["V_std_rel"] = V.std(1) / (V.mean(1) + 1e-9)
    d["I_std_rel"] = I.std(1) / (I.mean(1) + 1e-9)
    d["I_por_rpm"] = df.I_rms_mean / (r + 1e-9)                                # F10
    d["Z_ap"]      = df.V_rms_mean / (df.I_rms_mean + 1e-9)
    d["P_ap"]      = df.V_rms_mean * df.I_rms_mean * np.sqrt(3) / 1000.0
    d["P_por_rpm"] = d["P_ap"] / (r + 1e-9)

    d["dT_wind"]  = df.temp_winding__mean - df.Tamb                            # contra ambiente
    d["dT_b1"]    = df.temp_bearing1__mean - df.Tamb
    d["dT_b2"]    = df.temp_bearing2__mean - df.Tamb
    d["dT_bear"]  = df.temp_bearing1__mean - df.temp_bearing2__mean            # F06
    d["dT_w_b"]   = df.temp_winding__mean - df[["temp_bearing1__mean", "temp_bearing2__mean"]].mean(1)
    d["dT_w_P"]   = d["dT_wind"] / (d["P_ap"] + 1e-9)                          # F11
    d["dT_b_rpm"] = d[["dT_b1", "dT_b2"]].max(1) / (r + 1e-9)

    d["head_n"]   = df.delta_p_mean / r2                                       # ley de afinidad
    d["flow_n"]   = df.flow_mean / (r + 1e-9)
    d["p_ratio"]  = df.pressure_out_mean / (df.pressure_in_mean + 1e-9)
    d["p_in_n"]   = df.pressure_in_mean / r2                                   # F12
    d["hidr_pot"] = df.flow_mean * df.delta_p_mean / (r ** 3 + 1e-9)
    d["rend_ap"]  = (df.flow_mean * df.delta_p_mean) / (d["P_ap"] + 1e-9)      # F13
    d["flow_p"]   = df.flow_mean / (df.delta_p_mean.abs() + 1e-3)

    d["rpm_cv"] = df.rpm_std / (df.rpm_mean + 1e-9)
    return d.replace([np.inf, -np.inf], np.nan)

def zscore_maquina(df, base_df, cols):
    """Desviación respecto de la propia máquina. Sólo usa features y machine_id."""
    tmp = base_df[["machine_id"]].join(df[cols])
    g = tmp.groupby("machine_id")[cols]
    z = (tmp[cols] - g.transform("median")) / (g.transform("std") + 1e-9)
    z.columns = [c + "__z" for c in cols]
    return z

def construir(df, modo="fisicas_z"):
    if modo == "crudas":
        return df[BASE].copy()
    if modo == "crudas_z":
        return pd.concat([df[BASE], zscore_maquina(df, df, BASE)], axis=1)
    X = pd.concat([df[BASE], fisicas(df)], axis=1)
    if modo == "fisicas":
        return X
    return pd.concat([X, zscore_maquina(X, df, list(X.columns))], axis=1)

for m in ["crudas", "crudas_z", "fisicas", "fisicas_z"]:
    print("%-10s %s" % (m, construir(train, m).shape))

## 4. Benchmark: features × esquema de validación

Cuatro conjuntos de features, los dos esquemas, LightGBM en ambos para que la
comparación sea limpia.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
import lightgbm as lgb

def LGB(seed=RANDOM_STATE):
    return lgb.LGBMClassifier(
        objective="multiclass", num_class=14, n_estimators=600, learning_rate=0.05,
        num_leaves=31, min_child_samples=20, subsample=0.9, subsample_freq=1,
        colsample_bytree=0.8, reg_lambda=1.0, verbose=-1, random_state=seed,
        n_jobs=N_JOBS, deterministic=True, force_col_wise=True)

MAQ = train.machine_id.to_numpy()
SES = train.session_id.to_numpy()

def evaluar(proba13, idx):
    e = pd.DataFrame({"window_id": train.window_id.iloc[idx].to_numpy()})
    for j, f in enumerate(FAULT_IDS):
        e[f] = np.clip(proba13[:, j], 0, 1)
    return compute_score(train.iloc[idx], e)["overall"]

def cv_agrupada(X, ctor, semillas=(0, 1, 2)):
    """Esquema optimista: máquina nueva, mismo régimen."""
    acc = np.zeros((len(X), 14))
    for s in semillas:
        oof = np.zeros((len(X), 14))
        for a, b in StratifiedGroupKFold(5, shuffle=True, random_state=s).split(X, estado, MAQ):
            m = ctor(RANDOM_STATE + s); m.fit(X.iloc[a], estado[a])
            oof[b] = m.predict_proba(X.iloc[b])
        acc += oof
    return acc / len(semillas)

def cv_regimen(X, ctor):
    """Esquema conservador: máquina nueva Y régimen nuevo."""
    m = ctor(RANDOM_STATE); m.fit(X.iloc[i_lentas], estado[i_lentas])
    return m.predict_proba(X.iloc[i_rapidas])

TODOS = np.arange(len(train))
filas = []
for modo in ["crudas", "crudas_z", "fisicas", "fisicas_z"]:
    X = construir(train, modo)
    a = evaluar(cv_agrupada(X, LGB)[:, 1:], TODOS)["final_score"]
    b = evaluar(cv_regimen(X, LGB)[:, 1:], i_rapidas)["final_score"]
    filas.append(dict(features=modo, n_cols=X.shape[1], group_kfold=a, cambio_regimen=b))
    print("  %-10s  GroupKFold=%.2f   régimen=%.2f" % (modo, a, b))

tabla_feats = pd.DataFrame(filas)
display(tabla_feats.style.format({"group_kfold": "{:.2f}", "cambio_regimen": "{:.2f}"}).hide(axis="index"))

Las features físicas ganan en los dos esquemas. El z-score por máquina suma
encima de ellas: los dos ataques son distintos y **se acumulan**.

Una advertencia sobre el z-score que conviene registrar: da una ventaja clara en
`GroupKFold` (+2,4) y bastante menor bajo cambio de régimen. No hay que
venderlo como que resuelve la generalización a máquinas nuevas — lo que sí hace
es alinear las distribuciones, y eso se puede medir aparte.

In [ ]:
from scipy.stats import ks_2samp

Xtr_z, Xte_z = construir(train, "fisicas_z"), construir(test, "fisicas_z")
cols_z = [c for c in Xtr_z.columns if c.endswith("__z")]
cols_o = [c[:-3] for c in cols_z]

ks_o = [ks_2samp(Xtr_z[c].dropna(), Xte_z[c].dropna()).statistic for c in cols_o]
ks_z = [ks_2samp(Xtr_z[c].dropna(), Xte_z[c].dropna()).statistic for c in cols_z]
print("KS medio train vs test:  sin normalizar = %.3f   normalizado = %.3f"
      % (np.mean(ks_o), np.mean(ks_z)))
print("features con KS > 0.2 :  sin normalizar = %d       normalizado = %d"
      % (sum(k > .2 for k in ks_o), sum(k > .2 for k in ks_z)))

from scipy.stats import gaussian_kde
CRIT = ["rpm_mean", "delta_p_mean", "flow_mean", "voltage_a__rms"]
fig, axes = plt.subplots(len(CRIT), 2, figsize=(11, 2.3 * len(CRIT)))
for i, c in enumerate(CRIT):
    for j, col in enumerate([c, c + "__z"]):
        a, b = Xtr_z[col].dropna().values, Xte_z[col].dropna().values
        gr = np.linspace(min(a.min(), b.min()), max(a.max(), b.max()), 200)
        axes[i, j].fill_between(gr, gaussian_kde(a)(gr), color="#0E4F5C", alpha=.35, label="train")
        axes[i, j].fill_between(gr, gaussian_kde(b)(gr), color="#9A5B00", alpha=.35, label="test")
        axes[i, j].set_title("%s  (KS=%.3f)" % (col, ks_2samp(a, b).statistic), fontsize=9)
        axes[i, j].set_yticks([]); axes[i, j].spines[["top", "right"]].set_visible(False)
axes[0, 0].legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

## 5. Ensambles y promedio por sesión

El promedio por sesión se apoya en el hecho verificado en la sección 1: las 7
ventanas de una sesión comparten etiqueta. Promediarlas elimina ruido de ventana
sin usar nada prohibido — sólo `session_id`, que viene en el test.

Se usa media geométrica porque estamos promediando probabilidades y penaliza
más los desacuerdos fuertes.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

MODELOS = {
    "lgb": LGB,
    "et":  lambda s=RANDOM_STATE: make_pipeline(SimpleImputer(strategy="median"),
              ExtraTreesClassifier(n_estimators=600, min_samples_leaf=2,
                                   max_features="sqrt", n_jobs=N_JOBS, random_state=s)),
    "rf":  lambda s=RANDOM_STATE: make_pipeline(SimpleImputer(strategy="median"),
              RandomForestClassifier(n_estimators=600, min_samples_leaf=2,
                                     max_features="sqrt", n_jobs=N_JOBS, random_state=s)),
    "lr":  lambda s=RANDOM_STATE: make_pipeline(SimpleImputer(strategy="median"),
              StandardScaler(), LogisticRegression(max_iter=2000, C=0.5, random_state=s)),
}

def promedio_sesion(P, idx):
    """Media geométrica dentro de cada sesión. Sólo usa session_id."""
    d = pd.DataFrame(P); d["s"] = SES[idx]
    g = np.exp(d.groupby("s").transform(lambda v: np.log(np.clip(v, 1e-9, 1)).mean())).to_numpy()
    return g / g.sum(axis=1, keepdims=True)

X = construir(train, "fisicas_z")
oof_m, reg_m = {}, {}
for n, ctor in MODELOS.items():
    oof_m[n] = cv_agrupada(X, ctor)
    reg_m[n] = cv_regimen(X, ctor)
    print("  %s entrenado" % n)

In [ ]:
COMBOS = [("lgb",), ("lgb", "et"), ("lgb", "rf", "lr"),
          ("lgb", "et", "rf", "lr"), ("lgb", "et", "lr"), ("lgb", "et", "rf")]

def mezclar(d, combo):
    P = np.mean([d[n] for n in combo], axis=0)
    return P / P.sum(axis=1, keepdims=True)

res = []
for combo in COMBOS:
    A, B = mezclar(oof_m, combo), mezclar(reg_m, combo)
    res.append(dict(
        ensamble="+".join(combo),
        group_kfold=evaluar(A[:, 1:], TODOS)["final_score"],
        gkf_sesion=evaluar(promedio_sesion(A, TODOS)[:, 1:], TODOS)["final_score"],
        regimen=evaluar(B[:, 1:], i_rapidas)["final_score"],
        reg_sesion=evaluar(promedio_sesion(B, i_rapidas)[:, 1:], i_rapidas)["final_score"]))

tabla_ens = pd.DataFrame(res)
display(tabla_ens.style.format({c: "{:.2f}" for c in tabla_ens.columns[1:]})
        .background_gradient(subset=["reg_sesion"], cmap="BuGn").hide(axis="index"))

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.6))
x = np.arange(len(tabla_ens))
ax.scatter(tabla_ens.gkf_sesion, x, s=70, color="#9A5B00", label="GroupKFold (optimista)")
ax.scatter(tabla_ens.reg_sesion, x, s=70, color="#0E4F5C", label="cambio de régimen (conservador)")
for i in x:
    ax.plot([tabla_ens.reg_sesion[i], tabla_ens.gkf_sesion[i]], [i, i],
            color="#D3DBDF", zorder=0, lw=2)
ax.set_yticks(x); ax.set_yticklabels(tabla_ens.ensamble)
ax.axvline(43.42, color="#8F2A1D", ls="--", lw=1.2)
ax.text(43.42, -0.7, " servidor: 43,42", color="#8F2A1D", fontsize=9)
ax.set_xlabel("final_score"); ax.legend(fontsize=8, frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False); ax.invert_yaxis()
plt.tight_layout(); plt.show()

**El orden entre los dos esquemas no coincide, y esa es la lección del
notebook.** `lgb+rf+lr` es el mejor bajo `GroupKFold` (54,44) y de los peores
bajo cambio de régimen (45,43). Era la recomendación de la versión anterior, y
es la misma trampa que ya nos costó la brecha entre 48 y 43,42.

`lgb+et+rf` es cuarto en el esquema optimista y **primero en el conservador**
(47,43). Se elige ése. La regla que queda para el resto de la jornada: **ante
desacuerdo entre los dos esquemas, gana el conservador**, porque es el único que
reproduce la condición del test.

## 6. Modelo final y entrega

Se reentrena sobre el train completo y se predice el test. Todo con `n_jobs=1`
para que el submit sea reproducible.

In [ ]:
COMBO_FINAL = ("lgb", "et", "rf")

X_train_f = construir(train, "fisicas_z")
X_test_f  = construir(test,  "fisicas_z")
X_test_f  = X_test_f.reindex(columns=X_train_f.columns)   # mismo orden de columnas

pred = []
for n in COMBO_FINAL:
    m = MODELOS[n](RANDOM_STATE)
    m.fit(X_train_f, estado)
    pred.append(m.predict_proba(X_test_f))
    print("  %s entrenado sobre el train completo" % n)

P_test = np.mean(pred, axis=0)
P_test = P_test / P_test.sum(axis=1, keepdims=True)

# promedio por sesión sobre el test
d = pd.DataFrame(P_test); d["s"] = test.session_id.to_numpy()
P_test = np.exp(d.groupby("s").transform(lambda v: np.log(np.clip(v, 1e-9, 1)).mean())).to_numpy()
P_test = P_test / P_test.sum(axis=1, keepdims=True)

submission = pd.DataFrame({"window_id": test.window_id.to_numpy()})
for j, f in enumerate(FAULT_IDS):
    submission[f] = np.clip(P_test[:, j + 1], 0, 1)

submission = validate_prediction_package(test, submission)
Path("outputs").mkdir(exist_ok=True)
ruta = Path("outputs") / "submit_fisicas_lgb_et_rf.csv"
submission.to_csv(ruta, index=False)

print("\narchivo :", ruta)
print("md5     :", hashlib.md5(ruta.read_bytes()).hexdigest())
print("filas   :", len(submission), "| esperadas:", len(test))
display(submission.head())

In [ ]:
# Chequeos antes de subir
P = submission[FAULT_IDS].to_numpy()
assert submission.shape == (len(test), 14)
assert list(submission.columns) == ["window_id"] + FAULT_IDS
assert submission.window_id.is_unique
assert set(submission.window_id) == set(test.window_id)
assert np.isfinite(P).all() and (P >= 0).all() and (P <= 1).all()
print("formato validado")

print("\nΣp media  : %.3f   (train: %.3f)" % (P.sum(1).mean(), estado.astype(bool).mean()))
print("Σp p05-p95: %.3f - %.3f" % tuple(np.percentile(P.sum(1), [5, 95])))

acciones = pd.Series([scoring.choose_action({f: float(P[i, j])
                      for j, f in enumerate(FAULT_IDS)})[0] + "/" +
                      str(scoring.choose_action({f: float(P[i, j])
                      for j, f in enumerate(FAULT_IDS)})[1] or "")
                      for i in range(len(P))])
print("\nacciones que induce la entrega:")
print(acciones.value_counts().to_string())

## 7. Conclusión y qué sigue

**El techo tabular está en 47–48**, con el estimador conservador. La brecha
contra el 43,42 del envío anterior se explica por dos cosas medidas acá: las
features físicas y la elección de ensamble bajo el esquema correcto.

Estimación para este envío: **46–48**. Se apoya en dos anclas independientes que
coinciden — el estimador por cambio de régimen dice 47,43, y aplicar al
`GroupKFold` la misma brecha observada en el envío anterior da ~46,7. No prometo
los 52 que sugeriría leer el `GroupKFold` de frente.

Lo que queda, en orden:

1. **Las señales crudas.** La accuracy de familia se estanca en 0,68 y
   `cost_score` —el 70 % del score— paga exactamente por ella. Las seis fallas
   mecánicas se confunden entre sí porque la tabla trae un solo armónico por
   canal (1x y 2x); BPFO, BPFI y BSF están en la señal, no en la tabla.
2. **La hipótesis de los combos.** Si el test tiene fallas simultáneas, el
   softmax las subestima. Se puede sondear sin etiquetas mirando la
   distribución de Σp y la entropía sobre `train_unlabeled`.
3. **Calibración**, que es lo más barato que queda sin features nuevas.

### Registro para el probatorio

In [ ]:
REGISTRO = [
    dict(tipo="hallazgo", texto="El CV agrupado por máquina sobrestima",
         evidencia="crudas+z: GroupKFold 46,65 / régimen 40,53 / servidor 43,42",
         decision="se agrega validación por cambio de régimen como juez conservador"),
    dict(tipo="hallazgo", texto="El tamaño efectivo es 250 sesiones, no 1750 ventanas",
         evidencia="7 ventanas por sesión, etiqueta constante en las 250",
         decision="promedio geométrico por sesión en la entrega"),
    dict(tipo="hallazgo", texto="El test opera en otro régimen",
         evidencia="rpm 1742->2052, delta_p 0,51->2,61 (5x)",
         decision="features adimensionales normalizadas por rpm"),
    dict(tipo="decision", texto="Features físicas de dominio",
         evidencia="crudas 37,56 -> físicas 40,67 -> físicas+z 42,57 bajo régimen",
         decision="se adoptan; los dos ataques se acumulan"),
    dict(tipo="descarte", texto="Ensamble lgb+rf+lr",
         evidencia="mejor en GroupKFold (54,44) y de los peores bajo régimen (45,43)",
         decision="descartado por la regla: ante desacuerdo gana el conservador"),
    dict(tipo="hallazgo", texto="El baseline de la cátedra no es pasivo",
         evidencia="con las prevalencias, INSPECT mecánica (7,30) < MONITOR (8,76)",
         decision="la vara real es acertar la familia más del 46 % de las veces"),
    dict(tipo="decision", texto="n_jobs=1 en todos los modelos",
         evidencia="con n_jobs=-1 dos corridas dan hashes distintos",
         decision="requisito del art. 5: el submit tiene que ser reproducible"),
]
registro = pd.DataFrame(REGISTRO)
display(registro)
registro.to_json("registro_v3.json", orient="records", force_ascii=False, indent=2)